# MCP Security Architecture Review Workshop

This capstone reviews an exact multi-tenant support-agent change. We will inspect representations, prove control claims with current evidence, enumerate protected paths, execute abuse cases, record an independently attested outcome, and invalidate it after material change. Nothing in the notebook authorizes production.

## Decision boundary

Models and participants may propose threats, findings, and remediation. Trusted application code validates package identity, evidence, paths, test outcomes, reviewer authority, recording, and reassessment. `eligible_for_risk_decision` is not risk acceptance or deployment approval.

In [ ]:
import runpy
from dataclasses import replace
from datetime import timedelta
m = runpy.run_path('lab.py')
NOW = m['NOW']
sorted(name for name in ('ArchitectureReviewer', 'ReviewRegistry', 'ArchitecturePackage', 'ThreatScenario') if name in m)

## 1. Scope the exact change

The support agent is adding a write-capable tool and adopting MCP `2026-07-28`. The package pins owners, tenants, data classes, components, flows, protocol revisions, assumptions, non-goals, and a change ID. The digest makes later mutation visible.

In [ ]:
package = m['demo_package']()
{
    'package_id': package.package_id,
    'version': package.version,
    'protocol_versions': package.protocol_versions,
    'tenants': sorted(set().union(*(item.tenant_scope for item in package.components))),
    'digest': package.package_digest,
}

### Component inventory

A component name is not a control. Inspect exact owner, version, digest, trust zone, tenant scope, and capabilities.

In [ ]:
[
    {
        'id': item.component_id, 'kind': item.kind, 'zone': item.trust_zone,
        'version': item.version, 'capabilities': sorted(item.capabilities),
    }
    for item in package.components
]

### Data and authority flows

Every edge names protocol, data class, authentication, encryption, target authorization, tenant binding, telemetry, and side-effect behavior. These are review assertions that still need evidence and tests.

In [ ]:
[
    {
        'flow': flow.flow_id, 'path': f'{flow.source} -> {flow.target}',
        'protocol': flow.protocol, 'class': flow.data_classification,
        'target_authz': flow.authorized_at_target, 'tenant_bound': flow.tenant_bound,
        'side_effect': flow.side_effect,
    }
    for flow in package.flows
]

## 2. Enumerate protected paths

The lab uses NetworkX rather than visually assuming traffic follows the intended route. Every entrypoint-to-resource path needs complete per-hop protection, multiple independent enforcers, and final-resource authorization.

In [ ]:
graph = m['nx'].DiGraph()
graph.add_nodes_from(item.component_id for item in package.components)
graph.add_edges_from((flow.source, flow.target) for flow in package.flows)
[path for start in package.entrypoints for end in package.protected_resources for path in m['nx'].all_simple_paths(graph, start, end)]

## 3. Replace checklist booleans with falsifiable claims

The capstone covers identity, authorization, provenance, isolation, egress, data protection, telemetry, incident response, recovery, and change management. Each claim names an owner, exact subject, and required evidence kinds.

In [ ]:
claims = m['demo_claims'](package)
[{'claim': item.claim_id, 'subject': item.subject_id, 'proof': sorted(item.required_evidence_kinds)} for item in claims]

## 4. Admit exact current evidence

Passing evidence counts only when its claim, subject/version, producer, locator, test IDs, times, artifact digest, and evidence digest are valid. Multiple required kinds remain separate denominator slots.

In [ ]:
evidence = m['demo_evidence'](package, claims)
{
    'items': len(evidence),
    'kinds': sorted({item.kind for item in evidence}),
    'all_integrity_valid': all(item.evidence_digest == m['digest'](item.unsigned()) for item in evidence),
}

## 5. Turn threats into executable abuse cases

Framework mapping structures the analysis but does not prove exposure. Each scenario binds entrypoint, target, protected properties, ATT&CK/ATLAS references, and one abuse-test ID.

In [ ]:
threats = m['demo_threats']()
[{'threat': item.title, 'path': (item.entrypoint, item.target), 'refs': item.attack_refs, 'test': item.abuse_test_id} for item in threats]

### Independent outcome proof

A denial response is not a pass if a forbidden effect committed. The accepted result is exact-package, authorized-tenant, evidence-linked `pass + blocked_and_verified`.

In [ ]:
results = m['demo_abuse_results'](package, threats, evidence)
[{'test': item.test_id, 'result': item.result, 'effect': item.observed_effect, 'evidence': item.evidence_ids} for item in results]

## 6. Review the complete design

The complete fixture should have no findings and full disclosed coverage. Its terminal state is intentionally limited to eligibility for a separate risk decision.

In [ ]:
reviewer = m['ArchitectureReviewer']()
report = reviewer.review(package, claims, evidence, threats, results, NOW)
{
    'state': report.state, 'findings': len(report.findings),
    'evidence_coverage': report.metrics.evidence_coverage,
    'threat_test_coverage': report.metrics.threat_test_coverage,
    'critical_path_coverage': report.metrics.critical_path_coverage,
    'production_authorized': report.production_authorized,
}

## 7. Inject a final-resource authorization failure

The flawed fixture keeps the same gateway but removes target authorization and tenant binding from the server-to-ticket flow. This tests the common but unsafe assumption that gateway admission is sufficient.

In [ ]:
flawed_package = m['demo_package'](secure=False)
flawed_claims = m['demo_claims'](flawed_package)
flawed_evidence = m['demo_evidence'](flawed_package, flawed_claims)
flawed_threats = m['demo_threats']()
flawed_results = m['demo_abuse_results'](flawed_package, flawed_threats, flawed_evidence)
flawed_report = reviewer.review(flawed_package, flawed_claims, flawed_evidence, flawed_threats, flawed_results, NOW)
[(item.severity, item.code, item.flow_ids) for item in flawed_report.findings]

## 8. Missing recovery proof is not a clean review

Remove one required evidence kind. The denominator stays visible, coverage falls, and the claim produces a high finding.

In [ ]:
gap_evidence = m['demo_evidence'](package, claims, omit_kind='recovery_drill')
gap_results = m['demo_abuse_results'](package, threats, gap_evidence)
gap_report = reviewer.review(package, claims, gap_evidence, threats, gap_results, NOW)
{
    'state': gap_report.state,
    'coverage': gap_report.metrics.evidence_coverage,
    'slots': (gap_report.metrics.satisfied_evidence_slots, gap_report.metrics.evidence_slots),
    'codes': sorted({item.code for item in gap_report.findings}),
}

## 9. A failed abuse test overrides a coverage percentage

Coverage answers whether scenarios were tested. It does not mean the defenses passed. Here the test ran but the forbidden effect completed.

In [ ]:
failed_results = (replace(results[0], result=m['EvidenceResult'].FAIL, observed_effect='forbidden_effect_completed'),) + results[1:]
failed_report = reviewer.review(package, claims, evidence, threats, failed_results, NOW)
{
    'coverage': failed_report.metrics.threat_test_coverage,
    'defense_pass_rate': failed_report.metrics.abuse_defense_pass_rate,
    'finding': [(item.severity, item.code) for item in failed_report.findings],
}

## 10. Independently attest the exact review

Package owners cannot attest their own review. The short-lived signature binds purpose, package digest, report digest, reviewer identity, role, issuance, and expiry.

In [ ]:
registry = m['ReviewRegistry']()
attestation = registry.attest(package, report, 'reviewer:independent', 'security_architect', NOW + timedelta(minutes=1))
{
    'purpose': attestation.purpose, 'package_digest': attestation.package_digest,
    'report_digest': attestation.report_digest, 'expires_at': attestation.expires_at,
}

### Idempotent recording, not deployment

A stable operation ID makes an exact retry return the committed record. The attestation is atomic and single-use across different operations. The record still cannot authorize production.

In [ ]:
recorded = registry.record('operation:capstone-review', package, report, attestation, NOW + timedelta(minutes=2))
retried = registry.record('operation:capstone-review', package, report, attestation, NOW + timedelta(minutes=2))
{'same_record': recorded == retried, 'state': recorded.state, 'production_authorized': recorded.production_authorized}

## 11. Material change invalidates the conclusion

A new capability changes the reviewed subject and assumptions. The change event must match the package ID and prior digest and occur after recording.

In [ ]:
change = m['MaterialChange'](
    'change:new-write-tool', package.package_id, package.package_digest,
    frozenset({'capability', 'authorization'}), NOW + timedelta(days=1),
)
invalidated = registry.invalidate('operation:capstone-review', change)
{'before': recorded.state, 'after': invalidated.state, 'production_authorized': invalidated.production_authorized}

## 12. Portfolio metrics preserve separate meanings

Evidence, threat-test, and path coverage use different populations. Eligible packages are workflow state, not a deployment-success rate. Critical findings remain counts rather than being averaged away.

In [ ]:
m['review_portfolio_metrics']([report, flawed_report])

## Common tooling and its limits

Use C4/DFD tooling, OWASP Threat Dragon or pytm, NetworkX/graph stores, OPA or Cedar, official MCP SDK utilities and Inspector, pytest/Hypothesis/fuzzers, OpenTelemetry/SIEM, Sigstore/SLSA/SBOM/VEX, and CI evidence stores as appropriate. Their output remains evidence to validate: a graph does not know business validity, policy analysis does not prove runtime enforcement, and generated threats cannot expand authority.

## Production upgrades

Replace in-memory records with transactional durable storage and optimistic versioning; admit signed evidence from governed producers; connect graph inventory to deployment/runtime discovery; execute backend-specific protocol, policy, tenant, egress, incident, and recovery tests; integrate findings with accountable workflow; monitor evidence expiry and material change; keep review, risk, exception, deployment, and execution authorities separate.

## Exercises

1. Add a malicious local stdio server and exact launch/isolation/egress evidence.
2. Insert a legacy-to-modern MCP adapter and test lifecycle downgrade and caller-context loss.
3. Add Beta tenant state and a shared cache; prove every partition and backup path.
4. Add an asynchronous write with timeout-after-commit and durable reconciliation.
5. Add a signed vulnerable dependency and separate provenance, risk, exception, and admission.
6. Create a material protocol change and identify which evidence is reusable versus invalid.

## Reflection and final checkpoint

Can your review package answer, with exact evidence: what system/version/change is in scope; which paths reach protected resources; where independent authorization occurs; which threats were actually tested; whether forbidden effects were independently absent; who owns every gap; what decision the review may and may not support; and which material event invalidates it?